In [1]:
#generate 3D structure using the ALATIS naming convention using InChI string as input 
#helpful for visualization, not necessarily correct geometry

from rdkit import Chem
from rdkit.Chem import AllChem

def generate_alatis_3d_structure(inchi_string, output_filename="alatis_3D_optimized.pdb"):
    """
    Generates a high-quality 3D structure directly from an InChI string and labels 
    the atoms using the canonical InChI nomenclature to match your generic mapping dictionary.
    """
    
    print("Parsing InChI and generating graph...")
    mol_3d = Chem.MolFromInchi(inchi_string)
    if mol_3d is None:
        print("Error: Invalid InChI string.")
        return
        
    mol_3d = Chem.AddHs(mol_3d)
    
    print("Embedding 3D coordinates (ETKDGv3) and optimizing (MMFF94)...")
    # ETKDGv3 generates the initial 3D conformer
    embed_code = AllChem.EmbedMolecule(mol_3d, AllChem.ETKDGv3())
    if embed_code == -1:
        print("Error: 3D coordinate embedding failed.")
        return
        
    # MMFF94 forcefield relaxes the geometry to realistic bond lengths/angles
    AllChem.MMFFOptimizeMolecule(mol_3d)

    print("Applying canonical nomenclature to PDB atoms...")
    # Because mol_3d was generated from InChI, its enumeration order inherently 
    # matches the reference indices used in your dictionary mapping script.
    for idx, atom in enumerate(mol_3d.GetAtoms()):
        canonical_name = f"{atom.GetSymbol()}{idx}"
        
        # Configure PDB Info block
        pdb_info = Chem.AtomPDBResidueInfo()
        # PDB atom names must be up to 4 characters, left-justified
        pdb_info.SetName(f"{canonical_name:<4}") 
        pdb_info.SetResidueName("LIG")
        pdb_info.SetResidueNumber(1)
        
        atom.SetMonomerInfo(pdb_info)

    # Save the final structure
    Chem.MolToPDBFile(mol_3d, output_filename)
    print(f"Success! 3D structure saved to {output_filename}.")

  

In [2]:
#D-cellobiose
INCHI_STR = "InChI=1S/C12H22O11/c13-1-3-5(15)6(16)9(19)12(22-3)23-10-4(2-14)21-11(20)8(18)7(10)17/h3-20H,1-2H2/t3-,4-,5-,6+,7-,8-,9-,10-,11-,12+/m1/s1"

generate_alatis_3d_structure(INCHI_STR, "cellobiose_3D_ALATIS.pdb")

Parsing InChI and generating graph...
Embedding 3D coordinates (ETKDGv3) and optimizing (MMFF94)...
Applying canonical nomenclature to PDB atoms...
Success! 3D structure saved to cellobiose_3D_ALATIS.pdb.


In [3]:
#Maltose

INCHI_STR = "InChI=1S/C12H22O11/c13-1-3-5(15)6(16)9(19)12(22-3)23-10-4(2-14)21-11(20)8(18)7(10)17/h3-20H,1-2H2/t3-,4-,5-,6+,7-,8-,9-,10-,11-,12-/m1/s1"

generate_alatis_3d_structure(INCHI_STR, "maltose_3D_ALATIS.pdb")

Parsing InChI and generating graph...
Embedding 3D coordinates (ETKDGv3) and optimizing (MMFF94)...
Applying canonical nomenclature to PDB atoms...
Success! 3D structure saved to maltose_3D_ALATIS.pdb.


In [4]:
# GlcNAc-OME 
INCHI_STR = "InChI=1S/C9H17NO6/c1-4(12)10-6-8(14)7(13)5(3-11)16-9(6)15-2/h5-9,11,13-14H,3H2,1-2H3,(H,10,12)/t5-,6-,7-,8-,9-/m1/s1"
    
generate_alatis_3d_structure(INCHI_STR, "GlcNAc-OME_3D_ALATIS.pdb")

Parsing InChI and generating graph...
Embedding 3D coordinates (ETKDGv3) and optimizing (MMFF94)...
Applying canonical nomenclature to PDB atoms...
Success! 3D structure saved to GlcNAc-OME_3D_ALATIS.pdb.


In [5]:
#D-Glucose
INCHI_STR = "InChI=1S/C6H12O6/c7-1-2-3(8)4(9)5(10)6(11)12-2/h2-11H,1H2/t2-,3-,4+,5-,6+/m1/s1"

generate_alatis_3d_structure(INCHI_STR, "glucose_3D_ALATIS.pdb")

Parsing InChI and generating graph...
Embedding 3D coordinates (ETKDGv3) and optimizing (MMFF94)...
Applying canonical nomenclature to PDB atoms...
Success! 3D structure saved to glucose_3D_ALATIS.pdb.


In [6]:
# Build a mapping dictionary between GLYCAM and ALATIS atom numbering using graph matching with the mol2 files

import sys
from rdkit import Chem
import pprint

def sanitize_mol2_for_rdkit(mol2_path):
    """Sanitizes GLYCAM forcefield atom types (e.g., 'Oh', 'Cg') to standard elements."""
    with open(mol2_path, 'r') as f:
        mol2_lines = f.readlines()
        
    cleaned_lines = []
    in_atom_block = False
    for line in mol2_lines:
        if line.startswith('@<TRIPOS>BOND'):
            in_atom_block = False
        if in_atom_block and line.strip():
            parts = line.split()
            if len(parts) >= 9:
                element = parts[5][0] # Truncate 'Oh' to 'O'
                line = f" {parts[0]:>5} {parts[1]:<8} {parts[2]:>9} {parts[3]:>9} {parts[4]:>9} {element:<5} {parts[6]:>4} {parts[7]:<8} {parts[8]}\n"
        if line.startswith('@<TRIPOS>ATOM'):
            in_atom_block = True
        cleaned_lines.append(line)
        
    return mol2_lines, "".join(cleaned_lines)

def generate_alatis_mapping(mol2_path, alatis_mol_path):
    """
    Maps GLYCAM custom atom names to true ALATIS IDs by aligning the connectivity 
    graph against an ALATIS-generated reference .mol file.
    """
    # 1. Extract raw GLYCAM names and load sanitized mol2 into RDKit
    raw_mol2_lines, clean_mol2_block = sanitize_mol2_for_rdkit(mol2_path)
    
    mol2_names = []
    in_atom_block = False
    for line in raw_mol2_lines:
        if line.startswith('@<TRIPOS>BOND'): break
        if in_atom_block and line.strip():
            parts = line.split()
            mol2_names.append(f"{parts[6]}_{parts[7]}_{parts[1]}") # e.g., "2_4GB_C1"
        if line.startswith('@<TRIPOS>ATOM'): in_atom_block = True

    glycam_mol = Chem.MolFromMol2Block(clean_mol2_block, removeHs=False)
    if glycam_mol is None:
        print("Error: RDKit could not parse the sanitized mol2 file.")
        return None

    # 2. Load the reference .mol file generated by your MATLAB `alatis_command`
    alatis_ref_mol = Chem.MolFromMolFile(alatis_mol_path, removeHs=False)
    if alatis_ref_mol is None:
        print(f"Error: Could not parse ALATIS reference file {alatis_mol_path}.")
        return None

    # 3. Perform Stereochemically-Aware Graph Isomorphism
    # Query: alatis_ref_mol, Target: glycam_mol
    match_indices = glycam_mol.GetSubstructMatch(alatis_ref_mol, useChirality=True)

    if not match_indices:
        print("Warning: Strict chiral match failed. Attempting topological match...")
        match_indices = glycam_mol.GetSubstructMatch(alatis_ref_mol, useChirality=False)
        
    if not match_indices:
        print("Error: The ALATIS reference graph does not match the mol2 connectivity.")
        return None

    # 4. Build the mapping dictionary
    mapping_dict = {}
    for alatis_idx, glycam_idx in enumerate(match_indices):
        if glycam_idx < len(mol2_names):
            glycam_identifier = mol2_names[glycam_idx]
            ref_atom = alatis_ref_mol.GetAtomWithIdx(alatis_idx)
            
            # The secret to ALATIS IDs: Element Symbol + (1-based Index in the file)
            true_alatis_id = f"{ref_atom.GetSymbol()}{alatis_idx + 1}"
            
            mapping_dict[glycam_identifier] = true_alatis_id

    return mapping_dict


In [7]:
#D-Cellubiose

#can be downloaded from GLYCAM cbBuilder 
MOL2_FILE = "../D-cellobiose/GLYCAM_min-gas.mol2" 

#can be downloaded directly form BMRB database or can be generated from InChI string
ALATIS_FILE = "../D-cellobiose/BMRB_bmse000138.mol" 

map_dict = generate_alatis_mapping(MOL2_FILE, ALATIS_FILE)
if map_dict:
    print("GLYCAM to True ALATIS Map:")
    pprint.pprint(map_dict)

GLYCAM to True ALATIS Map:
{'1_ROH_HO1': 'H45',
 '1_ROH_O1': 'O20',
 '2_4GB_C1': 'C11',
 '2_4GB_C2': 'C8',
 '2_4GB_C3': 'C7',
 '2_4GB_C4': 'C10',
 '2_4GB_C5': 'C4',
 '2_4GB_C6': 'C2',
 '2_4GB_H1': 'H36',
 '2_4GB_H2': 'H33',
 '2_4GB_H2O': 'H43',
 '2_4GB_H3': 'H32',
 '2_4GB_H3O': 'H42',
 '2_4GB_H4': 'H35',
 '2_4GB_H5': 'H29',
 '2_4GB_H61': 'H27',
 '2_4GB_H62': 'H26',
 '2_4GB_H6O': 'H39',
 '2_4GB_O2': 'O18',
 '2_4GB_O3': 'O17',
 '2_4GB_O4': 'O23',
 '2_4GB_O5': 'O21',
 '2_4GB_O6': 'O14',
 '3_0GB_C1': 'C12',
 '3_0GB_C2': 'C9',
 '3_0GB_C3': 'C6',
 '3_0GB_C4': 'C5',
 '3_0GB_C5': 'C3',
 '3_0GB_C6': 'C1',
 '3_0GB_H1': 'H37',
 '3_0GB_H2': 'H34',
 '3_0GB_H2O': 'H44',
 '3_0GB_H3': 'H31',
 '3_0GB_H3O': 'H41',
 '3_0GB_H4': 'H30',
 '3_0GB_H4O': 'H40',
 '3_0GB_H5': 'H28',
 '3_0GB_H61': 'H25',
 '3_0GB_H62': 'H24',
 '3_0GB_H6O': 'H38',
 '3_0GB_O2': 'O19',
 '3_0GB_O3': 'O16',
 '3_0GB_O4': 'O15',
 '3_0GB_O5': 'O22',
 '3_0GB_O6': 'O13'}


[16:19:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.


In [8]:
#D-Cellubiose .mol file generated

MOL2_FILE = "../D-cellobiose/GLYCAM_min-gas.mol2" 
    
ALATIS_FILE = "../D-cellobiose/BMRB_bmse000138_generated.mol"

map_dict = generate_alatis_mapping(MOL2_FILE, ALATIS_FILE)
if map_dict:
    print("GLYCAM to True ALATIS Map:")
    pprint.pprint(map_dict)

GLYCAM to True ALATIS Map:
{'1_ROH_HO1': 'H45',
 '1_ROH_O1': 'O20',
 '2_4GB_C1': 'C11',
 '2_4GB_C2': 'C8',
 '2_4GB_C3': 'C7',
 '2_4GB_C4': 'C10',
 '2_4GB_C5': 'C4',
 '2_4GB_C6': 'C2',
 '2_4GB_H1': 'H36',
 '2_4GB_H2': 'H33',
 '2_4GB_H2O': 'H43',
 '2_4GB_H3': 'H32',
 '2_4GB_H3O': 'H42',
 '2_4GB_H4': 'H35',
 '2_4GB_H5': 'H29',
 '2_4GB_H61': 'H27',
 '2_4GB_H62': 'H26',
 '2_4GB_H6O': 'H39',
 '2_4GB_O2': 'O18',
 '2_4GB_O3': 'O17',
 '2_4GB_O4': 'O23',
 '2_4GB_O5': 'O21',
 '2_4GB_O6': 'O14',
 '3_0GB_C1': 'C12',
 '3_0GB_C2': 'C9',
 '3_0GB_C3': 'C6',
 '3_0GB_C4': 'C5',
 '3_0GB_C5': 'C3',
 '3_0GB_C6': 'C1',
 '3_0GB_H1': 'H37',
 '3_0GB_H2': 'H34',
 '3_0GB_H2O': 'H44',
 '3_0GB_H3': 'H31',
 '3_0GB_H3O': 'H41',
 '3_0GB_H4': 'H30',
 '3_0GB_H4O': 'H40',
 '3_0GB_H5': 'H28',
 '3_0GB_H61': 'H25',
 '3_0GB_H62': 'H24',
 '3_0GB_H6O': 'H38',
 '3_0GB_O2': 'O19',
 '3_0GB_O3': 'O16',
 '3_0GB_O4': 'O15',
 '3_0GB_O5': 'O22',
 '3_0GB_O6': 'O13'}


[16:19:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.


In [9]:
#Maltose .mol file generated

MOL2_FILE = "../Maltose/GLYCAM_min-gas.mol2" 
    
ALATIS_FILE = "../Maltose/BMRB_bmse000946_generated.mol"

map_dict = generate_alatis_mapping(MOL2_FILE, ALATIS_FILE)
if map_dict:
    print("GLYCAM to True ALATIS Map:")
    pprint.pprint(map_dict)

GLYCAM to True ALATIS Map:
{'1_ROH_HO1': 'H45',
 '1_ROH_O1': 'O20',
 '2_4GA_C1': 'C11',
 '2_4GA_C2': 'C8',
 '2_4GA_C3': 'C7',
 '2_4GA_C4': 'C10',
 '2_4GA_C5': 'C4',
 '2_4GA_C6': 'C2',
 '2_4GA_H1': 'H36',
 '2_4GA_H2': 'H33',
 '2_4GA_H2O': 'H43',
 '2_4GA_H3': 'H32',
 '2_4GA_H3O': 'H42',
 '2_4GA_H4': 'H35',
 '2_4GA_H5': 'H29',
 '2_4GA_H61': 'H27',
 '2_4GA_H62': 'H26',
 '2_4GA_H6O': 'H39',
 '2_4GA_O2': 'O18',
 '2_4GA_O3': 'O17',
 '2_4GA_O4': 'O23',
 '2_4GA_O5': 'O21',
 '2_4GA_O6': 'O14',
 '3_0GA_C1': 'C12',
 '3_0GA_C2': 'C9',
 '3_0GA_C3': 'C6',
 '3_0GA_C4': 'C5',
 '3_0GA_C5': 'C3',
 '3_0GA_C6': 'C1',
 '3_0GA_H1': 'H37',
 '3_0GA_H2': 'H34',
 '3_0GA_H2O': 'H44',
 '3_0GA_H3': 'H31',
 '3_0GA_H3O': 'H41',
 '3_0GA_H4': 'H30',
 '3_0GA_H4O': 'H40',
 '3_0GA_H5': 'H28',
 '3_0GA_H61': 'H25',
 '3_0GA_H62': 'H24',
 '3_0GA_H6O': 'H38',
 '3_0GA_O2': 'O19',
 '3_0GA_O3': 'O16',
 '3_0GA_O4': 'O15',
 '3_0GA_O5': 'O22',
 '3_0GA_O6': 'O13'}


[16:19:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.


In [10]:
#Lewis_a 

MOL2_FILE = "../Lewis_a/GLYCAM_min-gas.mol2" 

ALATIS_FILE = "../Lewis_a/BMRB_21031_generated.mol" 

map_dict = generate_alatis_mapping(MOL2_FILE, ALATIS_FILE)
if map_dict:
    print("GLYCAM to True ALATIS Map:")
    pprint.pprint(map_dict)

GLYCAM to True ALATIS Map:
{'1_OME_CH3': 'C3',
 '1_OME_H1': 'H44',
 '1_OME_H2': 'H45',
 '1_OME_H3': 'H46',
 '1_OME_O': 'O32',
 '2_WYB_C1': 'C19',
 '2_WYB_C2': 'C10',
 '2_WYB_C2N': 'C7',
 '2_WYB_C3': 'C18',
 '2_WYB_C4': 'C17',
 '2_WYB_C5': 'C9',
 '2_WYB_C6': 'C5',
 '2_WYB_CME': 'C2',
 '2_WYB_H1': 'H63',
 '2_WYB_H1M': 'H43',
 '2_WYB_H2': 'H54',
 '2_WYB_H2M': 'H42',
 '2_WYB_H2N': 'H66',
 '2_WYB_H3': 'H62',
 '2_WYB_H3M': 'H41',
 '2_WYB_H4': 'H61',
 '2_WYB_H5': 'H53',
 '2_WYB_H61': 'H50',
 '2_WYB_H62': 'H49',
 '2_WYB_H6O': 'H68',
 '2_WYB_N2': 'N22',
 '2_WYB_O2N': 'O25',
 '2_WYB_O3': 'O37',
 '2_WYB_O4': 'O36',
 '2_WYB_O5': 'O35',
 '2_WYB_O6': 'O24',
 '3_0fA_C1': 'C20',
 '3_0fA_C2': 'C15',
 '3_0fA_C3': 'C13',
 '3_0fA_C4': 'C11',
 '3_0fA_C5': 'C6',
 '3_0fA_C6': 'C1',
 '3_0fA_H1': 'H64',
 '3_0fA_H2': 'H59',
 '3_0fA_H2O': 'H73',
 '3_0fA_H3': 'H57',
 '3_0fA_H3O': 'H71',
 '3_0fA_H4': 'H55',
 '3_0fA_H4O': 'H69',
 '3_0fA_H5': 'H51',
 '3_0fA_H61': 'H38',
 '3_0fA_H62': 'H39',
 '3_0fA_H63': 'H40',
 '3_

[16:19:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.


In [11]:
#Sucrose

MOL2_FILE = "../Sucrose/GLYCAM_min-gas.mol2" 

ALATIS_FILE = "../../Example_NMRCouplingSimulation_NMRdb_and_GLYCAMdb/Sucrose/sucrose_generated_mol.mol" 

map_dict = generate_alatis_mapping(MOL2_FILE, ALATIS_FILE)
if map_dict:
    print("GLYCAM to True ALATIS Map:")
    pprint.pprint(map_dict)

GLYCAM to True ALATIS Map:
{'1_1GA_C1': 'C11',
 '1_1GA_C2': 'C9',
 '1_1GA_C3': 'C8',
 '1_1GA_C4': 'C6',
 '1_1GA_C5': 'C4',
 '1_1GA_C6': 'C1',
 '1_1GA_H1': 'H37',
 '1_1GA_H2': 'H35',
 '1_1GA_H2O': 'H44',
 '1_1GA_H3': 'H34',
 '1_1GA_H3O': 'H43',
 '1_1GA_H4': 'H32',
 '1_1GA_H4O': 'H41',
 '1_1GA_H5': 'H30',
 '1_1GA_H61': 'H25',
 '1_1GA_H62': 'H24',
 '1_1GA_H6O': 'H38',
 '1_1GA_O1': 'O23',
 '1_1GA_O2': 'O19',
 '1_1GA_O3': 'O18',
 '1_1GA_O4': 'O16',
 '1_1GA_O5': 'O21',
 '1_1GA_O6': 'O13',
 '2_0CU_C1': 'C3',
 '2_0CU_C2': 'C12',
 '2_0CU_C3': 'C10',
 '2_0CU_C4': 'C7',
 '2_0CU_C5': 'C5',
 '2_0CU_C6': 'C2',
 '2_0CU_H11': 'H29',
 '2_0CU_H12': 'H28',
 '2_0CU_H1O': 'H40',
 '2_0CU_H3': 'H36',
 '2_0CU_H3O': 'H45',
 '2_0CU_H4': 'H33',
 '2_0CU_H4O': 'H42',
 '2_0CU_H5': 'H31',
 '2_0CU_H61': 'H27',
 '2_0CU_H62': 'H26',
 '2_0CU_H6O': 'H39',
 '2_0CU_O1': 'O15',
 '2_0CU_O3': 'O20',
 '2_0CU_O4': 'O17',
 '2_0CU_O5': 'O22',
 '2_0CU_O6': 'O14'}


[16:19:28] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.


In [12]:
# Generate InChI string from mol2 file 

from rdkit import Chem

def sanitize_mol2_for_rdkit(mol2_path):
    """
    Sanitizes GLYCAM forcefield atom types (e.g., 'Oh', 'Cg') to standard 
    elements so RDKit can successfully parse the topology.
    """
    with open(mol2_path, 'r') as f:
        mol2_lines = f.readlines()
        
    cleaned_lines = []
    in_atom_block = False
    for line in mol2_lines:
        if line.startswith('@<TRIPOS>BOND'):
            in_atom_block = False
        if in_atom_block and line.strip():
            parts = line.split()
            if len(parts) >= 9:
                # Extract just the first letter (e.g., 'Oh' -> 'O')
                element = parts[5][0] 
                line = f" {parts[0]:>5} {parts[1]:<8} {parts[2]:>9} {parts[3]:>9} {parts[4]:>9} {element:<5} {parts[6]:>4} {parts[7]:<8} {parts[8]}\n"
        if line.startswith('@<TRIPOS>ATOM'):
            in_atom_block = True
        cleaned_lines.append(line)
        
    return "".join(cleaned_lines)

def mol2_to_inchi(mol2_path):
    """Converts a GLYCAM .mol2 file into a canonical InChI string."""
    
    # 1. Sanitize the GLYCAM atom types
    clean_mol2_block = sanitize_mol2_for_rdkit(mol2_path)
    
    # 2. Load the molecule
    mol = Chem.MolFromMol2Block(clean_mol2_block, removeHs=False)
    if mol is None:
        print("Error: RDKit could not parse the mol2 file.")
        return None

    # 3. Explicitly perceive stereochemistry from the 3D coordinates
    # This ensures your alpha/beta linkages are encoded in the InChI
    Chem.AssignStereochemistryFrom3D(mol)

    # 4. Generate the InChI string
    inchi_str = Chem.MolToInchi(mol)
    
    return inchi_str


In [13]:
# generate InChI string for D-Cellubiose
MOL2_FILE = "../D-cellobiose/GLYCAM_min-gas.mol2" 

inchi = mol2_to_inchi(MOL2_FILE)
if inchi:
    print(f"Generated InChI:\n{inchi}")

Generated InChI:
InChI=1S/C12H22O11/c13-1-3-5(15)6(16)9(19)12(22-3)23-10-4(2-14)21-11(20)8(18)7(10)17/h3-20H,1-2H2/t3-,4-,5-,6+,7-,8-,9-,10-,11-,12+/m1/s1


In [14]:
# generate InChI string for Maltose
MOL2_FILE = "../Maltose/GLYCAM_min-gas.mol2" 

inchi = mol2_to_inchi(MOL2_FILE)
if inchi:
    print(f"Generated InChI:\n{inchi}")

Generated InChI:
InChI=1S/C12H22O11/c13-1-3-5(15)6(16)9(19)12(22-3)23-10-4(2-14)21-11(20)8(18)7(10)17/h3-20H,1-2H2/t3-,4-,5-,6+,7-,8-,9-,10-,11+,12-/m1/s1


In [15]:
# generate InChI string for Lewis_a
MOL2_FILE = "../Lewis_a/GLYCAM_min-gas.mol2" 

inchi = mol2_to_inchi(MOL2_FILE)
if inchi:
    print(f"Generated InChI:\n{inchi}")

Generated InChI:
InChI=1S/C21H37NO15/c1-6-11(26)13(28)15(30)20(33-6)36-17-9(5-24)35-19(32-3)10(22-7(2)25)18(17)37-21-16(31)14(29)12(27)8(4-23)34-21/h6,8-24,26-31H,4-5H2,1-3H3/q-1/t6-,8+,9+,10+,11+,12-,13+,14-,15-,16+,17+,18+,19+,20-,21-/m0/s1


[16:19:28] WARNING: Accepted unusual valence(s): C(3)



In [16]:
# generate InChI string for Glucose
MOL2_FILE = "../DGlcpb1-OH/GLYCAM_DGlcpb1-OH/521a56ff-9a2e-4ded-bff5-730b6fbda659/Requested_Builds/structure/min-gas.mol2" 

inchi = mol2_to_inchi(MOL2_FILE)
if inchi:
    print(f"Generated InChI:\n{inchi}")

Generated InChI:
InChI=1S/C6H12O6/c7-1-2-3(8)4(9)5(10)6(11)12-2/h2-11H,1H2/t2-,3-,4+,5-,6-/m1/s1


In [17]:
# generate InChI string for Sucrose
MOL2_FILE = "../Sucrose/GLYCAM_min-gas.mol2" 

inchi = mol2_to_inchi(MOL2_FILE)
if inchi:
    print(f"Generated InChI:\n{inchi}")

Generated InChI:
InChI=1S/C12H22O11/c13-1-4-6(16)8(18)9(19)11(21-4)23-12(3-15)10(20)7(17)5(2-14)22-12/h4-11,13-20H,1-3H2/t4-,5-,6-,7-,8+,9-,10+,11-,12+/m1/s1


In [18]:
from rdkit import Chem

def clean_and_load_glycam_mol2(file_path: str):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    cleaned_lines = []
    in_atom_block = False
    
    for line in lines:
        if "@<TRIPOS>ATOM" in line:
            in_atom_block = True
            cleaned_lines.append(line)
            continue
        elif line.startswith("@<TRIPOS>"):
            in_atom_block = False
            
        if in_atom_block:
            parts = line.split()
            # A valid Tripos ATOM row has at least 6 properties: 
            # ID, Name, X, Y, Z, AtomType, [SubstID, SubstName, Charge]
            if len(parts) >= 6:
                atom_id = parts[0]
                atom_name = parts[1]
                x, y, z = parts[2], parts[3], parts[4]
                atom_type = parts[5]
                rest = parts[6:] if len(parts) > 6 else []
                
                # Strip GLYCAM/AMBER specific types to standard elements
                # Examples: 'Oh' -> 'O', 'Ho' -> 'H', 'Os' -> 'O', 'Cg' -> 'C'
                clean_type = atom_type
                if atom_type.startswith(('O', 'o')):
                    clean_type = 'O'
                elif atom_type.startswith(('C', 'c')):
                    clean_type = 'C'
                elif atom_type.startswith(('H', 'h')):
                    clean_type = 'H'
                elif atom_type.startswith(('N', 'n')):
                    clean_type = 'N'
                
                # Reconstruct the line cleanly with tabs so RDKit columns align perfectly
                remainder_str = "\t".join(rest)
                line = f"{atom_id}\t{atom_name}\t{x}\t{y}\t{z}\t{clean_type}\t{remainder_str}\n"
                    
        cleaned_lines.append(line)
        
    mol2_block = "".join(cleaned_lines)
    return Chem.MolFromMol2Block(mol2_block)

In [19]:
# Run the function on your file
mol = clean_and_load_glycam_mol2('../D-cellobiose/GLYCAM_min-gas.mol2')

if mol is not None:
    print("Canonical SMILES:")
    print(Chem.MolToSmiles(mol))
else:
    print("Failed to read the sanitized structural block.")


Canonical SMILES:
OC[C@H]1O[C@@H](O[C@H]2[C@H](O)[C@@H](O)[C@H](O)O[C@@H]2CO)[C@H](O)[C@@H](O)[C@@H]1O


In [20]:
# Run the function on your file
mol = clean_and_load_glycam_mol2('../DGlcpb1-OH/GLYCAM_DGlcpb1-OH/521a56ff-9a2e-4ded-bff5-730b6fbda659/Requested_Builds/structure/min-gas.mol2')

if mol is not None:
    print("Canonical SMILES:")
    print(Chem.MolToSmiles(mol))
else:
    print("Failed to read the sanitized structural block.")

Canonical SMILES:
OC[C@H]1O[C@@H](O)[C@H](O)[C@@H](O)[C@@H]1O


In [21]:
# Run the function on your file
mol = clean_and_load_glycam_mol2('../Sucrose/GLYCAM_min-gas.mol2')

if mol is not None:
    print("Canonical SMILES:")
    print(Chem.MolToSmiles(mol))
else:
    print("Failed to read the sanitized structural block.")

Canonical SMILES:
OC[C@H]1O[C@@](CO)(O[C@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]2O)[C@@H](O)[C@@H]1O


In [22]:
# Create a dictionary between the GLYCAM and BMRB STAR file where the ALATIS convention is not used in the BMRB 
